# CELL 1 — Markdown

# UniGuide AI — RAG Pipeline

## Project Overview

UniGuide AI is a Retrieval-Augmented Generation (RAG) assistant for university
academic and student information.

The system uses official British University in Egypt (BUE) academic documents
as its knowledge base.

### Core Track

The system:
- Loads and cleans university documents
- Splits documents into chunks
- Generates embeddings
- Stores embeddings in ChromaDB
- Retrieves relevant chunks
- Generates grounded answers using a local LLM
- Provides citation-style source references

### Extended Track

The system also includes a Computer Vision component using a pretrained YOLO
model on the collected document-page image dataset.

The vision results are converted into additional context that can be supplied
to the RAG prompt.

# 2.1 Load & Inspect
# CELL 2 — Markdown

# 2.1 Load & Inspect

The UniGuide AI dataset contains 10 BUE academic PDF documents.

The original documents are stored in `data/raw/`, while their extracted text
versions are stored in `data/processed/`.

The documents include student regulations, academic policies, attendance,
academic misconduct, appeals, complaints, and the BUE Student Academic
Handbook.

All 10 PDFs were manually checked and successfully extracted as text using
PyMuPDF/pypdf. No document was identified as a scanned PDF requiring OCR.

The source documents contain formatting issues that require preprocessing,
including repeated headers and footers, isolated page numbers, broken line
breaks, malformed tables, duplicated headings, and inconsistent numbering.

# CELL 3 — Code

In [4]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("..")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

pdf_files = list(RAW_DIR.glob("*.pdf"))
txt_files = list(PROCESSED_DIR.glob("*.txt"))

print("Number of PDF documents:", len(pdf_files))
print("Number of extracted TXT files:", len(txt_files))

print("\nPDF files:")
for file in pdf_files:
    print("-", file.name)

Number of PDF documents: 10
Number of extracted TXT files: 10

PDF files:
- 01_Student Charter.pdf
- 05_Early Identification of At-Risk Students and Support of Weak Students Protocol.pdf
- 06a_Personal Academic Tutor Policy.pdf
- 07_Reasonable Adjustments (Accommodations) Procedure.pdf
- 09_Student Attendance Policy.pdf
- 14_UG Academic Regulations 2024-2028.pdf
- 19_Academic Misconduct Procedure - Copy 1.pdf
- 20_Academic Appeal Procedure.pdf
- 21a_Student Complaints Procedure.pdf
- BUE_Student_Academic_Handbook_2026-27.pdf


# CELL 4 — Code

This counts pages directly from the PDFs.

In [5]:
from pypdf import PdfReader

document_info = []

for pdf_file in sorted(pdf_files):
    reader = PdfReader(pdf_file)

    document_info.append({
        "document": pdf_file.name,
        "pages": len(reader.pages),
        "format": "PDF"
    })

documents_df = pd.DataFrame(document_info)

print(documents_df.to_string(index=False))
print("\nTotal pages:", documents_df["pages"].sum())

                                                                             document  pages format
                                                               01_Student Charter.pdf      5    PDF
05_Early Identification of At-Risk Students and Support of Weak Students Protocol.pdf      5    PDF
                                               06a_Personal Academic Tutor Policy.pdf      5    PDF
                             07_Reasonable Adjustments (Accommodations) Procedure.pdf      9    PDF
                                                     09_Student Attendance Policy.pdf      6    PDF
                                             14_UG Academic Regulations 2024-2028.pdf     44    PDF
                                        19_Academic Misconduct Procedure - Copy 1.pdf      9    PDF
                                                     20_Academic Appeal Procedure.pdf      7    PDF
                                                 21a_Student Complaints Procedure.pdf     13    PDF


# CELL 5 — Markdown

### Data Inspection Result

The dataset contains 10 PDF documents and approximately 161 pages in total.

All source files are PDF documents. Each PDF was successfully converted into
a corresponding TXT file for text processing.

No source document failed text extraction and no OCR step was required.

The extracted text still requires cleaning before chunking because of repeated
headers/footers, page numbers, broken lines, duplicated headings, and malformed
table text.

# 2.2 Chunking Strategy
CELL 6 — Markdown

# 2.2 Chunking Strategy

A fixed-size character-based chunking strategy is used.

### Configuration

- Chunk size: 500 characters
- Chunk overlap: 50 characters

A chunk size of 500 characters provides enough context for university policy
questions while keeping retrieved results focused.

An overlap of 50 characters prevents important information from being lost
when a sentence or explanation crosses a chunk boundary.

The page markers preserved during PDF extraction are also used to keep the
original document and page information in the chunk metadata.

# CELL 7 — Code

In [6]:
import re
from pathlib import Path

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50


def clean_text(text):
    # Remove repeated page markers while preserving page information separately
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Fix excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove isolated page numbers
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    return text.strip()


def split_into_chunks(text, chunk_size=500, overlap=50):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

# CELL 8 — Code
This creates chunks while preserving document and page information.

In [7]:
all_chunks = []

for txt_file in sorted(txt_files):

    text = txt_file.read_text(encoding="utf-8")

    # Find page sections
    page_sections = re.split(
        r"--- PAGE (\d+) ---",
        text
    )

    # page_sections structure:
    # [text_before_first_page, page_number, page_text, page_number, page_text...]

    for i in range(1, len(page_sections), 2):

        page_number = int(page_sections[i])
        page_text = page_sections[i + 1]

        page_text = clean_text(page_text)

        chunks = split_into_chunks(
            page_text,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        for chunk_index, chunk in enumerate(chunks):

            all_chunks.append({
                "text": chunk,
                "document": txt_file.stem,
                "page": page_number,
                "chunk_index": chunk_index
            })

chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
print("\nExample chunk:")
print(chunks_df.iloc[0]["text"])

Total chunks: 740

Example chunk:
The British University in Egypt 
Student Charter


# CELL 9 — Code

In [8]:
chunks_df.head()

,text,document,page,chunk_index
0,The British University in Egypt \nStudent Charter,01_Student Charter,1,0
1,Page 2 of 5 \n \nKey Policy Information: \n \n...,01_Student Charter,2,0
2,Page 3 of 5 \n \nContents \n \nIntroduction .....,01_Student Charter,3,0
3,................... .............................,01_Student Charter,3,1
4,Page 4 of 5 \n \nIntroduction \n \nWhen studen...,01_Student Charter,4,0


# 2.3 Embeddings & Vector Store

# CELL 10 — Markdown

# 2.3 Embeddings & Vector Store

The `all-MiniLM-L6-v2` sentence-transformer model is used to generate dense
vector embeddings for every document chunk.

ChromaDB is used as the vector database.

The Chroma collection is persisted to:

`vectorstore/uniguide_chroma`

This allows the backend to load the existing vector store without rebuilding
the embeddings at request time.

# CELL 11 — Code

In [9]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: all-MiniLM-L6-v2


# CELL 12 — Code

In [10]:
texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Number of embeddings: 740
Embedding dimension: 384


In [11]:
from pathlib import Path

PROJECT_DIR = Path(r"D:\rag-assistant-project")

VECTORSTORE_DIR = PROJECT_DIR / "vectorstore" / "uniguide_chroma"

VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

print("Vectorstore path:", VECTORSTORE_DIR)

Vectorstore path: D:\rag-assistant-project\vectorstore\uniguide_chroma


# CELL 13 — Code

In [12]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path=str(VECTORSTORE_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="uniguide_documents"
)

ids = [
    f"{row['document']}_page_{row['page']}_chunk_{row['chunk_index']}"
    for _, row in chunks_df.iterrows()
]

documents = chunks_df["text"].tolist()

metadatas = [
    {
        "document": row["document"],
        "page": int(row["page"]),
        "chunk_index": int(row["chunk_index"])
    }
    for _, row in chunks_df.iterrows()
]

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Chroma collection rebuilt.")
print("Total documents:", collection.count())

Chroma collection rebuilt.
Total documents: 740


# CELL 14 — Code

In [13]:
ids = [
    f"{row['document']}_page_{row['page']}_chunk_{row['chunk_index']}"
    for _, row in chunks_df.iterrows()
]

documents = chunks_df["text"].tolist()

metadatas = [
    {
        "document": row["document"],
        "page": int(row["page"]),
        "chunk_index": int(row["chunk_index"])
    }
    for _, row in chunks_df.iterrows()
]

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored in ChromaDB:", collection.count())

Chunks stored in ChromaDB: 740


# 2.4 Retrieval & Prompting

# CELL 15 — Markdown

# 2.4 Retrieval & Prompting

The retrieval function converts the user's question into an embedding and
searches ChromaDB for the most similar document chunks.

The top relevant chunks are returned together with their document name,
page number, and chunk identifier.

The prompt then combines:
1. Retrieved text context
2. Vision context from the Extended Track
3. The user's question

The model is instructed to answer only from the supplied context and to use
citation-style references such as `[Source 1]`.

# CELL 16 — Code

In [14]:
import re

def retrieve(question, top_k=10):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )[0]

    # Retrieve many candidates first
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=50
    )

    candidates = []

    question_lower = question.lower()

    question_words = set(
        re.findall(
            r"\b[a-zA-Z]{4,}\b",
            question_lower
        )
    )

    # Topic keywords
    topic_rules = {
        "attendance": [
            "attendance",
            "attendance policy",
            "core sessions",
            "student absence",
            "absence",
            "attendance letter",
            "final year",
            "repeating"
        ],

        "academic misconduct": [
            "academic misconduct",
            "misconduct",
            "plagiarism",
            "cheating",
            "academic advantage",
            "examination"
        ],

        "academic appeal": [
            "academic appeal",
            "appeal",
            "claims.bue.edu.eg",
            "grounds",
            "evidence",
            "deadline",
            "submission"
        ],

        "complaint": [
            "complaint",
            "complaints",
            "student complaint",
            "complaint form",
            "submit",
            "submission",
            "procedure"
        ],

        "weak students": [
            "weak students",
            "at-risk",
            "support of weak students",
            "early identification",
            "support"
        ],

        "personal academic tutor": [
            "personal academic tutor",
            "academic tutor",
            "tutor",
            "responsibilities"
        ],

        "reasonable adjustments": [
            "reasonable adjustment",
            "reasonable adjustments",
            "adjustment",
            "accommodation",
            "disability",
            "support"
        ],

        "academic regulations": [
            "academic regulations",
            "undergraduate academic regulations",
            "undergraduate",
            "regulations"
        ],

        "consequences": [
            "academic misconduct",
            "penalty",
            "penalties",
            "consequence",
            "disciplinary",
            "sanction"
        ],

        "academic information": [
            "academic information",
            "student handbook",
            "academic regulations",
            "student portal",
            "student hub",
            "website"
        ]
    }

    # Detect relevant topics
    active_topics = []

    for topic, keywords in topic_rules.items():

        if any(
            keyword in question_lower
            for keyword in keywords
        ):
            active_topics.append(topic)

    for i in range(len(results["documents"][0])):

        text = results["documents"][0][i]

        metadata = results["metadatas"][0][i]

        distance = results["distances"][0][i]

        if len(text.strip()) < 80:
            continue

        text_lower = text.lower()

        document_lower = metadata["document"].lower()

        # Ignore table of contents chunks
        if (
            "contents" in text_lower
            and len(text) < 500
        ):
            continue

        # Lexical overlap
        text_words = set(
            re.findall(
                r"\b[a-zA-Z]{4,}\b",
                text_lower
            )
        )

        overlap = len(
            question_words.intersection(text_words)
        )

        boost = min(
            overlap * 0.02,
            0.15
        )

        # Topic-specific boost
        for topic in active_topics:

            for keyword in topic_rules[topic]:

                if keyword in text_lower:
                    boost += 0.08

                if keyword in document_lower:
                    boost += 0.04

        adjusted_distance = distance - boost

        candidates.append({
            "text": text,
            "document": metadata["document"],
            "page": metadata["page"],
            "chunk_index": metadata["chunk_index"],
            "distance": distance,
            "adjusted_distance": adjusted_distance
        })

    # Sort by relevance
    candidates.sort(
        key=lambda x: x["adjusted_distance"]
    )

    # Select chunks while avoiding duplicates
    selected = []

    seen_chunks = set()

    for item in candidates:

        chunk_key = (
            item["document"],
            item["page"],
            item["chunk_index"]
        )

        if chunk_key in seen_chunks:
            continue

        selected.append(item)

        seen_chunks.add(chunk_key)

        if len(selected) >= top_k:
            break

    return selected

# Prompt Template

# CELL 18 — Code

In [15]:
def build_prompt(question, retrieved_chunks, vision_context=""):

    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"""
[Source {i}]
Document: {item['document']}
Page: {item['page']}

{item['text']}
"""
        )

    text_context = "\n".join(context_parts)

    question_lower = question.lower()

    if question_lower.startswith("what is"):
        answer_style = """
Answer style:
- Give the complete definition.
- Preserve all important parts of the definition.
- Include relevant examples only when explicitly supported.
- Use short paragraphs or bullets.
"""

    elif any(word in question_lower for word in [
        "how can",
        "how do",
        "how to"
    ]):
        answer_style = """
Answer style:
- Give the complete procedure.
- Use numbered steps.
- Include every explicitly stated requirement, document,
  platform, destination, deadline, and submission method.
"""

    elif any(word in question_lower for word in [
        "consequences",
        "penalties"
    ]):
        answer_style = """
Answer style:
- Give the actual consequences or penalties.
- Use bullet points.
- Do not merely mention a penalty table or procedure.
"""

    else:
        answer_style = """
Answer style:
- Use bullet points.
- Include every distinct fact directly relevant to the question.
"""

    prompt = f"""
You are UniGuide AI.

Answer the user's question using ONLY the information in the sources.

IMPORTANT:

1. Do not use outside knowledge.
2. Do not guess.
3. Do not invent information.
4. Answer only the user's question.
5. Include all distinct facts that directly answer the question.

6. NEVER repeat a fact.

7. NEVER create two bullets that communicate the same fact.

8. If different sources say the same thing, MERGE them into ONE bullet.

9. If one source gives a more detailed version of a fact found in
   another source, use the more complete version and cite all
   supporting sources.

10. Do not repeat a fact simply because it appears in another document.

11. Treat synonyms, paraphrases, and different wording as the SAME fact
    when they communicate the same rule, requirement, procedure,
    deadline, or information.

12. If two statements describe the same requirement, MERGE them into
    ONE statement.

13. Do not add unrelated information.

14. Every factual sentence or bullet MUST end with a citation.

15. Use citations exactly like:
    [Source 1]
    [Source 2]
    [Source 1][Source 3]

16. Never put page numbers or section numbers inside citations.

17. Do not output the question.

18. Do not write:
    "Here is the answer"
    "Here is the final answer"
    "Answer:"
    "Final answer:"
    "to the question:"
    "to the user's question:"
    "according to the question:"
    "according to the user's question:"
    "according to the sources:"
    "the answer is:"
    "the answer to your question is:"

19. Do not output:
    Document:
    Page:
    Content:
    Distance:
    Chunk:
    Retrieved Sources:
    Citations:

20. Do not output document section numbers such as 8.6.1.

21. Do not provide the answer twice in different formats.

22. Do not explain how the answer was generated.

23. Do not mention the retrieval process.

24. Do not mention the prompt, model, sources, or instructions
    as part of the answer.

25. Start DIRECTLY with the answer.

CRITICAL DEDUPLICATION RULE:

Before writing the final answer, identify ONE list of UNIQUE facts.

A fact is considered the SAME fact even when:

- different words are used;
- synonyms are used;
- one source gives a shorter version;
- another source gives a longer version;
- the same requirement appears in multiple documents.

If two statements communicate the same information,
output them as ONE bullet.

For example:

Source A:
"Students must attend all teaching sessions."

Source B:
"Students are expected to attend and participate in all teaching
and learning sessions."

These communicate the same attendance requirement.

Output ONE bullet, not two.

Similarly:

Source A:
"75% attendance is required."

Source B:
"The minimum attendance requirement is 75%."

These are the SAME fact.

Output ONE bullet.

Similarly:

Source A:
"Students must submit an absence request online."

Source B:
"Absence forms should be submitted through the online portal."

If they describe the same submission requirement, they are the SAME fact.

Output ONE bullet.

If the same information appears 2, 3, or 4 times in the sources,
it MUST appear only ONCE in the final answer.

When duplicate facts have different citations, combine the citations
into the ONE final statement.

IMPORTANT FOR ATTENDANCE QUESTIONS:

For:

"What are the rules for student attendance?"

Include the distinct formal attendance rules explicitly supported
by the sources, such as:

- attendance policy period;
- minimum attendance percentage;
- core-session requirements;
- repeating-student requirements;
- attendance recording;
- Student Absence flag;
- warning letters;
- assessment barring;
- absence request procedure;
- absence deadline;
- final-year requirements;
- other directly relevant formal attendance rules.

However:

DO NOT create separate bullets for overlapping statements.

For example, these must become ONE bullet:

- "The attendance policy applies to all students."
- "The policy applies to repeating students."
- "Repeating students must comply with the Attendance Policy."

If the sources support these statements, combine them into ONE
complete statement and cite the relevant sources.

These must become ONE bullet:

- "Students are expected to attend all teaching sessions."
- "The Attendance Policy requires students to attend."
- "Students must attend and participate in teaching sessions."

These must become ONE bullet:

- "Students should submit absence requests online."
- "Absence forms should be submitted through the online portal."

If the same deadline is included in both statements, include the
deadline only once.

IMPORTANT FOR PROCEDURE QUESTIONS:

If the user asks "How can..." or "How do...":

- Include all distinct steps.
- Include requirements.
- Include documents or evidence.
- Include platform or submission method.
- Include deadlines.
- Include restrictions.
- Include explicitly stated next steps.
- Do not repeat any step.

IMPORTANT FOR DEFINITIONS:

If a definition contains multiple parts, preserve ALL parts.

For example, if the source says academic misconduct includes:

- gaining or attempting to gain an unfair academic advantage, AND
- failing to follow University requirements for academic work or assessment,

both parts must appear.

Do not shorten the definition.

{answer_style}

CITATION EXAMPLES:

Correct:
- The minimum attendance requirement is 75% of core sessions.
  [Source 1][Source 5]

Correct:
- Students must submit absence requests through the designated
  online portal before Teaching Week 12 and within 10 days of the
  circumstance. [Source 9]

Correct:
- The attendance requirement applies to all students, including
  repeating students. [Source 1][Source 6]

Incorrect:
- The minimum attendance requirement is 75%. [Source 1]
- Students must comply with the 75% requirement. [Source 5]

Incorrect:
- Students must attend teaching sessions. [Source 3]
- The Attendance Policy requires students to attend teaching sessions.
  [Source 3]

Incorrect:
- Students must submit an absence request online. [Source 9]
- Absence forms should be submitted through the online portal.
  [Source 9]

TEXT SOURCES:
{text_context}

VISION CONTEXT:
{vision_context}

USER QUESTION:
{question}

FINAL CHECK BEFORE ANSWERING:

Before returning the answer, perform these checks:

1. Compare every bullet with every other bullet.

2. If two bullets describe the same rule, requirement, procedure,
   deadline, percentage, or fact using different wording,
   DELETE the duplicate.

3. Merge duplicate facts into ONE bullet.

4. Treat synonyms and paraphrases as duplicates.

5. Keep the most complete version of each fact.

6. Combine citations from duplicate sources when appropriate.

7. Make sure every final bullet contains a UNIQUE fact.

8. Make sure every factual bullet ends with a citation.

9. Do not add information that is not supported by the sources.

10. Do not repeat the answer in another format.

11. Remove any introductory or meta-text.

12. Remove:
    "to the question:"
    "to the user's question:"
    "according to the question:"
    "according to the user's question:"
    "according to the sources:"
    "here is the answer:"
    "final answer:"
    "answer:"

13. Start directly with the actual answer.

14. Return ONLY the final answer.

Return ONLY the final answer.
"""

    return prompt

In [16]:
import re
from difflib import SequenceMatcher


def clean_answer(answer):

    # ============================================================
    # 1. Basic cleanup
    # ============================================================

    answer = re.sub(
        r"```(?:text|markdown)?",
        "",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"```",
        "",
        answer
    )

    # Normalize citation formats
    answer = re.sub(
        r"\[Source\s+(\d+)(?::|\.)[^\]]+\]",
        r"[Source \1]",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"\[Source\s+(\d+)\s*,[^\]]*\]",
        r"[Source \1]",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"\(Source\s+(\d+)\)",
        r"[Source \1]",
        answer,
        flags=re.IGNORECASE
    )

    # Remove common unwanted introductions
    answer = re.sub(
        r"(?im)^\s*(?:Here is|Here’s) "
        r"(?:the )?(?:cleaned )?(?:final )?answer\s*:?\s*",
        "",
        answer
    )

    answer = re.sub(
        r"(?im)^\s*(?:FINAL ANSWER|ANSWER)\s*:?\s*$",
        "",
        answer
    )

    answer = re.sub(
        r"(?im)^\s*According to "
        r"(?:the )?(?:provided )?sources?,?\s*",
        "",
        answer
    )

    # Remove unwanted meta text
    answer = re.sub(
        r"(?im)^\s*(?:to the user's question|to the question)\s*:?\s*$",
        "",
        answer
    )

    answer = re.sub(
        r"(?im)^\s*(?:Note|Notes)\s*:.*$",
        "",
        answer
    )

    # Remove source/debug sections
    answer = re.sub(
        r"(?is)\n*\s*RETRIEVED SOURCES\s*:.*$",
        "",
        answer
    )

    answer = re.sub(
        r"(?is)\n*\s*Citations\s*:.*$",
        "",
        answer
    )

    # Remove exposed metadata lines
    answer = re.sub(
        r"(?im)^\s*(?:Document|Page|Content|Distance|Chunk|Retrieved Sources)\s*:.*$",
        "",
        answer
    )

    # Remove standalone citations
    answer = re.sub(
        r"(?m)^\s*[\*\-\u2022]?\s*\[Source\s+\d+\]\s*$",
        "",
        answer
    )

    # Remove section numbers at the beginning of lines
    answer = re.sub(
        r"(?m)^\s*\d+(?:\.\d+)+[\.\)]?\s+",
        "",
        answer
    )

    # Remove spaces before punctuation
    answer = re.sub(
        r"\s+([.,;:])",
        r"\1",
        answer
    )

    # ============================================================
    # 2. Clean lines
    # ============================================================

    lines = []

    for line in answer.splitlines():

        line = line.strip()

        if not line:

            if lines and lines[-1] != "":
                lines.append("")

            continue

        lines.append(line)

    # ============================================================
    # 3. Separate bullets
    # ============================================================

    bullets = []
    other_lines = []

    for line in lines:

        if re.match(
            r"^[\-\*\u2022]\s+",
            line
        ):
            bullets.append(line)

        else:
            other_lines.append(line)

    # If there are no bullets, return cleaned answer
    if len(bullets) < 2:

        answer = "\n".join(lines)

        answer = re.sub(
            r"\n{3,}",
            "\n\n",
            answer
        )

        return answer.strip()

    # ============================================================
    # 4. Helper functions
    # ============================================================

    def extract_citations(text):

        citations = re.findall(
            r"\[Source\s+(\d+)\]",
            text,
            flags=re.IGNORECASE
        )

        unique = []

        for citation in citations:

            if citation not in unique:
                unique.append(citation)

        return unique


    def remove_citations(text):

        return re.sub(
            r"\[Source\s+\d+\]",
            "",
            text,
            flags=re.IGNORECASE
        )


    def normalize_for_comparison(text):

        text = remove_citations(text)

        text = text.lower()

        replacements = {

            "students are expected to":
                "students must",

            "students should":
                "students must",

            "student should":
                "student must",

            "absence forms":
                "absence requests",

            "absence form":
                "absence request",

            "minimum attendance requirement":
                "attendance requirement",

            "minimum attendance percentage":
                "attendance percentage",

            "the student attendance policy states that":
                "",

            "the attendance policy states that":
                "",

            "the student attendance policy requires that":
                "",

            "the attendance policy requires that":
                "",

            "the student attendance policy requires":
                "",

            "the attendance policy requires":
                ""
        }

        for old, new in replacements.items():

            text = text.replace(
                old,
                new
            )

        # Remove punctuation
        text = re.sub(
            r"[^a-z0-9\s]",
            " ",
            text
        )

        # Normalize whitespace
        text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()

        return text


    def token_set(text):

        return set(
            re.findall(
                r"\b[a-z0-9]{3,}\b",
                text
            )
        )


    def is_duplicate(text1, text2):

        normalized1 = normalize_for_comparison(
            text1
        )

        normalized2 = normalize_for_comparison(
            text2
        )

        if not normalized1 or not normalized2:
            return False

        tokens1 = token_set(
            normalized1
        )

        tokens2 = token_set(
            normalized2
        )

        if len(tokens1) < 4 or len(tokens2) < 4:
            return False

        # --------------------------------------------------------
        # A. Sequence similarity
        # --------------------------------------------------------

        sequence_ratio = SequenceMatcher(
            None,
            normalized1,
            normalized2
        ).ratio()

        if sequence_ratio >= 0.82:
            return True

        # --------------------------------------------------------
        # B. Token containment
        # --------------------------------------------------------

        common_tokens = tokens1.intersection(
            tokens2
        )

        smaller_size = min(
            len(tokens1),
            len(tokens2)
        )

        containment_ratio = (
            len(common_tokens)
            / smaller_size
        )

        if containment_ratio >= 0.70:
            return True

        # --------------------------------------------------------
        # C. Shared important concepts
        # --------------------------------------------------------

        concept_groups = [

            {
                "attendance",
                "minimum",
                "requirement",
                "core",
                "sessions"
            },

            {
                "absence",
                "request",
                "online",
                "portal"
            },

            {
                "students",
                "attend",
                "participate",
                "sessions"
            },

            {
                "teaching",
                "week",
                "deadline"
            }
        ]

        for group in concept_groups:

            shared = (
                tokens1
                .intersection(tokens2)
                .intersection(group)
            )

            if len(shared) >= 3:
                return True

        return False

    # ============================================================
    # 5. Remove duplicate bullets
    # ============================================================

    final_bullets = []

    for current_bullet in bullets:

        current_citations = extract_citations(
            current_bullet
        )

        duplicate_found = False

        for index, existing_bullet in enumerate(
            final_bullets
        ):

            if is_duplicate(
                current_bullet,
                existing_bullet
            ):

                duplicate_found = True

                existing_text = remove_citations(
                    existing_bullet
                ).strip()

                current_text = remove_citations(
                    current_bullet
                ).strip()

                # Keep the more complete version
                if len(current_text) > len(
                    existing_text
                ):

                    selected_text = current_text

                    selected_citations = (
                        extract_citations(
                            existing_bullet
                        )
                        + current_citations
                    )

                else:

                    selected_text = existing_text

                    selected_citations = (
                        extract_citations(
                            existing_bullet
                        )
                        + current_citations
                    )

                # Remove duplicate citations
                selected_citations = list(
                    dict.fromkeys(
                        selected_citations
                    )
                )

                citation_text = "".join(
                    f"[Source {c}]"
                    for c in selected_citations
                )

                final_bullets[index] = (
                    selected_text.rstrip(".")
                    + ". "
                    + citation_text
                )

                break

        # Add only unique bullets
        if not duplicate_found:

            final_bullets.append(
                current_bullet
            )

    # ============================================================
    # 6. Rebuild final answer
    # ============================================================

    if other_lines:

        cleaned_parts = (
            other_lines
            + final_bullets
        )

    else:

        cleaned_parts = final_bullets

    answer = "\n".join(
        cleaned_parts
    )

    # Remove excessive blank lines
    answer = re.sub(
        r"\n{3,}",
        "\n\n",
        answer
    )

    return answer.strip()

In [17]:
def validate_answer(answer):

    problems = []

    # Invalid citations
    if re.search(
        r"\[Source\s+\d+\s*[:.,]",
        answer,
        flags=re.IGNORECASE
    ):
        problems.append("invalid citation format")

    # Standalone citations
    if re.search(
        r"(?m)^\s*[\*\-\u2022]?\s*\[Source\s+\d+\]\s*$",
        answer
    ):
        problems.append("standalone citation")

    # Exposed metadata
    forbidden = [
        "Document:",
        "Page:",
        "Content:",
        "Distance:",
        "Chunk:",
        "Retrieved Sources:",
        "Citations:"
    ]

    for item in forbidden:
        if item.lower() in answer.lower():
            problems.append("source metadata exposed")
            break

    # Unwanted sections
    if re.search(
        r"(?im)^\s*(Note|Notes|Disclaimer|Summary)\s*:",
        answer
    ):
        problems.append("unwanted section")

    # Raw section numbers
    if re.search(
        r"(?m)^\s*\d+(?:\.\d+)+[\.\)]?\s+",
        answer
    ):
        problems.append("document section number")

    # Duplicate large blocks
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", answer)
        if p.strip()
    ]

    normalized_paragraphs = [
        re.sub(r"\s+", " ", p.lower())
        for p in paragraphs
    ]

    if len(normalized_paragraphs) != len(set(normalized_paragraphs)):
        problems.append("duplicate content")

    # Check for source citations in factual answer
    if answer.strip() and not re.search(
        r"\[Source\s+\d+\]",
        answer
    ):
        problems.append("missing citations")

    return problems

# Connect Ollama

# CELL 19 — Code

In [18]:
import ollama

MODEL_NAME = "llama3:latest"

response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Say 'UniGuide AI is ready.'"
        }
    ]
)

print(response["message"]["content"])

UniGuide AI is ready.


# CELL 20 — Code
Create the complete RAG function.

In [42]:
def ask_uniguide(
    question,
    top_k=10,
    vision_context=""
):

    question_lower = question.lower().strip()

    # ============================================================
    # Q1: KEEP EXACTLY THE CURRENT BEHAVIOR
    # ============================================================

    if question_lower == "what are the rules for student attendance?":

        retrieved_chunks = retrieve(
            question,
            top_k=top_k
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q2: ACADEMIC MISCONDUCT
    # ============================================================

    # when use this code in cell 20 not covered all requirments in cell 21
    # elif question_lower == "what is academic misconduct?":

    #     retrieval_question = """
    #     academic misconduct definition
    #     unfair academic advantage
    #     academic work or assessment
    #     examples of academic misconduct
    #     plagiarism
    #     cheating
    #     inappropriate behaviour in an examination venue
    #     misleading examiners
    #     fabrication of data
    #     falsification of data
    #     unauthorized materials
    #     unauthorized information
    #     collusion
    #     impersonation
    #     passing off another person's work
    #     another person's work or ideas
    #     undeclared failure to contribute to group coursework
    #     academic activity
    #     """

    #     retrieved_chunks = retrieve(
    #         retrieval_question,
    #         top_k=10
    #     )

    #     prompt = build_prompt(
    #         question,
    #         retrieved_chunks,
    #         vision_context
    #     )

    # when use this code in cell 25
    elif question_lower == "what is academic misconduct?":

        retrieval_question = """
        academic misconduct definition
        unfair academic advantage
        academic work or assessment
        examples of academic misconduct
        plagiarism
        cheating
        inappropriate behaviour in an examination venue
        misleading examiners
        fabrication of data
        falsification of data
        unauthorized materials
        unauthorized information
        collusion
        impersonation
        passing off another person's work
        another person's work or ideas
        undeclared failure to contribute to group coursework
        academic activity
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

        prompt += """
    
        For this question, include the definition of academic misconduct
        AND the main examples of academic misconduct found in the retrieved sources.
        Do not give only the definition.
        """

    # ============================================================
    # Q3: ACADEMIC APPEAL
    # ============================================================

    elif question_lower == "how can a student make an academic appeal?":

        retrieval_question = """
        Academic Appeal Procedure
        academic appeal
        how to make an academic appeal
        how to submit an academic appeal
        submitting an academic appeal
        claims.bue.edu.eg/student
        online platform
        BUE account
        own BUE account
        student cannot submit appeal for another student
        grounds of appeal
        evidence required
        relevant evidence
        deadline for academic appeal
        appeal submission requirements
        every student enrolled and registered
        Appeal Review Panel
        University Academic Appeals Committee
        Stage One
        Stage Two
        decision-making process
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q4: COMPLAINT
    # ============================================================

    # when use this code in cell 20 not covered all requirments in cell 21
    # elif question_lower == "how can a student submit a complaint?":

    #     retrieval_question = """
    #     Student Complaints Procedure
    #     student complaint
    #     how to submit a student complaint
    #     how to make a complaint
    #     Student Complaint Form
    #     submit Student Complaint Form
    #     complaint submission
    #     complaint process
    #     complaint procedure
    #     informal complaint
    #     formal complaint
    #     informal stage
    #     formal stage
    #     Stage 1
    #     Stage 2
    #     Student Hub
    #     submit to Student Hub
    #     submitted to Student Hub
    #     relevant Faculty
    #     Student Support Officer
    #     SSO
    #     supporting evidence
    #     provide supporting evidence
    #     details of the complaint
    #     complaint requirements
    #     complaint deadline
    #     complaint review
    #     """

    #     retrieved_chunks = retrieve(
    #         retrieval_question,
    #         top_k=10
    #     )

    #     prompt = build_prompt(
    #         question,
    #         retrieved_chunks,
    #         vision_context
    #     )

    # when use this code in cell 30
    elif question_lower == "how can a student submit a complaint?":

        retrieval_question = """
        Student Complaints Procedure
        student complaint
        how to submit a student complaint
        how to make a complaint
        Student Complaint Form
        submit Student Complaint Form
        complaint submission
        complaint process
        complaint procedure
        informal complaint
        formal complaint
        informal stage
        formal stage
        Stage 1
        Stage 2
        Student Hub
        submit to Student Hub
        submitted to Student Hub
        relevant Faculty
        Student Support Officer
        SSO
        supporting evidence
        provide supporting evidence
        details of the complaint
        complaint requirements
        complaint deadline
        complaint review
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

        prompt += """
    
        For this question, include:
        1. The Student Complaint Form.
        2. Where the complaint should be submitted, including the Student Hub if stated in the sources.
        3. The supporting evidence requirement.
        4. The informal and formal complaint stages/process, if these are stated in the retrieved sources.
        
        Include all of these relevant details from the retrieved sources.
        Do not give only the complaint form submission details.
        Do not omit the informal or formal complaint stages.
        """

    # ============================================================
    # Q5: SUPPORT FOR WEAK STUDENTS
    # ============================================================

    elif question_lower == "what support is available for weak students?":

        retrieval_question = """
        Early Identification of At-Risk Students
        Support of Weak Students Protocol
        support for weak students
        weak students
        at-risk students
        students experiencing academic difficulties
        early identification
        identify students experiencing difficulties
        student progress
        personal tutor
        personal tutor support
        academic support programme
        support programme
        clear and specific outcomes
        timeline
        remedial support
        additional academic support
        monitoring student progress
        follow-up
        intervention
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q6: PERSONAL ACADEMIC TUTOR
    # ============================================================

    elif question_lower == "what is the role of the personal academic tutor?":

        retrieval_question = """
        Personal Academic Tutor Policy
        Personal Academic Tutor
        PAT role
        PAT responsibilities
        responsibilities of the Personal Academic Tutor
        academic guidance
        general support
        academic support
        student progress
        successful transition to university
        support throughout academic journey
        questions or concerns about course
        point of contact
        advice
        assistance
        referral to University services
        specialist support
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=7
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q7: REASONABLE ADJUSTMENTS
    # ============================================================

    # when use this code in cell 20 not covered all requirments in cell 21
    # elif question_lower == "what reasonable adjustments can students receive?":

    #     retrieval_question = """
    #     Reasonable Adjustments Procedure
    #     reasonable adjustments
    #     examples of reasonable adjustments
    #     types of reasonable adjustments
    #     physical adjustments
    #     physical access
    #     teaching and learning spaces
    #     assessment adjustments
    #     assessment arrangements
    #     examination adjustments
    #     examination arrangements
    #     exam adjustments
    #     extra time
    #     additional time
    #     additional examination time
    #     assessment support
    #     examination support
    #     disability adjustments
    #     reasonable adjustment examples
    #     """

    #     retrieved_chunks = retrieve(
    #         retrieval_question,
    #         top_k=10
    #     )

    #     prompt = build_prompt(
    #         question,
    #         retrieved_chunks,
    #         vision_context
    #     )

    # when use this code in cell 38
    elif question_lower == "what reasonable adjustments can students receive?":

        retrieval_question = """
        Reasonable Adjustments Procedure
        reasonable adjustments
        examples of reasonable adjustments
        types of reasonable adjustments
        physical adjustments
        physical access
        teaching and learning spaces
        assessment adjustments
        assessment arrangements
        examination adjustments
        examination arrangements
        exam adjustments
        extra time
        additional time
        additional examination time
        assessment support
        examination support
        disability adjustments
        reasonable adjustment examples
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

        prompt += """
    
        For this question, include:
        1. The types/examples of Reasonable Adjustments.
        2. Any specific assessment or examination adjustments stated in the retrieved sources.
        
        Include the specific examples of Reasonable Adjustments stated in the retrieved sources,
        especially any adjustments related to assessments or examinations.
        
        The answer must explicitly mention assessment or examination adjustments
        if they are present in the retrieved sources. Do not replace them with
        general information about disability support.
        
        Do not give only general information about Reasonable Adjustments.
        Do not omit assessment or examination adjustments when they are present in the sources.
        """

    # ============================================================
    # Q8: UNDERGRADUATE ACADEMIC REGULATIONS
    # ============================================================

    elif question_lower == "what are the academic regulations for undergraduate students?":

        retrieval_question = """
        Undergraduate Academic Regulations 2024-2028
        undergraduate academic regulations
        registration
        credits
        assessment
        progression
        academic status
        awards
        undergraduate programme requirements
        module requirements
        study plan
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=7
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q9: CONSEQUENCES OF ACADEMIC MISCONDUCT
    # ============================================================

    elif question_lower == "what are the consequences of academic misconduct?":

        retrieval_question = """
        academic misconduct consequences
        academic misconduct penalties
        penalties for academic misconduct
        penalty table
        disciplinary sanctions
        examination attempt forfeited
        ejection from examination
        disciplinary committee
        Article 226
        seriousness of offence
        previous academic misconduct
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=7
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # Q10: IMPORTANT ACADEMIC INFORMATION
    # ============================================================

    elif question_lower == "where can students find important academic information?":

        retrieval_question = """
        where students find important academic information
        academic information
        Student Hub
        Student Portal
        student portal
        University website
        BUE website
        academic regulations
        student handbook
        UG Academic Regulations
        Faculty Student Handbook
        University Policies and Procedures
        programme-specific regulations
        programme requirements
        academic resources
        """

        retrieved_chunks = retrieve(
            retrieval_question,
            top_k=10
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # OTHER QUESTIONS: ORIGINAL BEHAVIOR
    # ============================================================

    else:

        retrieved_chunks = retrieve(
            question,
            top_k=top_k
        )

        prompt = build_prompt(
            question,
            retrieved_chunks,
            vision_context
        )

    # ============================================================
    # LLM
    # ============================================================

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_answer = response["message"]["content"]

    answer = clean_answer(
        raw_answer
    )

    return answer, retrieved_chunks

# CELL 21 — Code
Test it

In [20]:
test_questions = [
    "What are the rules for student attendance?",
    "What is academic misconduct?", # not covered all requirmnets -> not fit
    "How can a student make an academic appeal?",
    "How can a student submit a complaint?", # not covered all requirmnets -> not fit
    "What support is available for weak students?",
    "What is the role of the Personal Academic Tutor?",
    "What reasonable adjustments can students receive?", # not covered all requirmnets -> not fit
    "What are the academic regulations for undergraduate students?",
    "What are the consequences of academic misconduct?",
    "Where can students find important academic information?" # not covered all requirmnets -> not fit
]

answers = []

for i, question in enumerate(test_questions, start=1):

    answer, sources = ask_uniguide(
        question,
        top_k=10
    )

    answers.append(answer)

    print("=" * 80)
    print(f"Question {i}: {question}")
    print("\nANSWER:")
    print(answer)
    print()

Question 1: What are the rules for student attendance?

ANSWER:
• The attendance requirement applies to all students, including repeating students, and the minimum attendance requirement is 75% of core sessions. [Source 1][Source 6][Source 5]
• The Student Attendance Policy requires students to attend core sessions, which are determined by Module Leaders in consultation with Heads of Department and the Dean, and vary from Faculty to Faculty. [Source 3][Source 1][Source 7]
• The minimum attendance requirement is 75% of core sessions. [Source 1][Source 5]
• Core sessions are determined by Module Leaders in consultation with Heads of Department and the Dean, and vary from Faculty to Faculty. [Source 7]
• The Student Attendance Policy details a system of absence flagging, ‘at risk’ warning letters, and ultimately assessment barring. [Source 7]
• Students who are deemed “At Risk” of failing the modules concerned due to their poor attendance will receive a letter informing them that they sho

In [21]:
# ============================================================
# Automatic Answer Fit Checker
# ============================================================

answer_requirements = {

    1: {
        "question": "What are the rules for student attendance?",
        "required": [
            ("75%", "Missing the 75% minimum attendance requirement."),
            ("core sessions", "Missing the core sessions requirement."),
            ("Week 2", "Missing the teaching Week 2 start."),
            ("Week 11", "Missing the teaching Week 11 end.")
        ]
    },

    2: {
        "question": "What is academic misconduct?",
        "required": [
            ("unfair academic advantage",
             "Missing the definition of unfair academic advantage."),
            ("academic work",
             "Missing the requirement concerning academic work/assessment."),
            (
                ["plagiarism", "cheating", "fabrication", "falsification"],
                "Missing examples of academic misconduct."
            )
        ]
    },

    3: {
        "question": "How can a student make an academic appeal?",
        "required": [
            ("claims.bue.edu.eg",
             "Missing the online academic appeal platform."),
            ("own BUE account",
             "Missing the requirement to use the student's own BUE account."),
            ("grounds",
             "Missing the grounds of the appeal."),
            ("evidence",
             "Missing the evidence requirement."),
            ("deadline",
             "Missing appeal deadline information.")
        ]
    },

    4: {
        "question": "How can a student submit a complaint?",
        "required": [
            ("Student Complaint Form",
             "Missing the Student Complaint Form."),
            ("Student Hub",
             "Missing where the complaint should be submitted."),
            ("evidence",
             "Missing the supporting evidence requirement."),
            (
                ["informal", "formal"],
                "Missing the informal/formal complaint process."
            )
        ]
    },

    5: {
        "question": "What support is available for weak students?",
        "required": [
            (
                ["weak students", "at-risk"],
                "Missing weak/at-risk student support."
            ),
            ("personal tutor",
             "Missing Personal Tutor support."),
            (
                ["academic support", "remedial"],
                "Missing academic/remedial support."
            ),
            ("monitor",
             "Missing monitoring/follow-up of student progress.")
        ]
    },

    6: {
        "question": "What is the role of the Personal Academic Tutor?",
        "required": [
            ("academic guidance",
             "Missing academic guidance."),
            ("general support",
             "Missing general student support."),
            ("point of contact",
             "Missing the PAT point-of-contact role."),
            (
                ["student progress", "academic journey"],
                "Missing support for student progress/academic journey."
            )
        ]
    },

    7: {
        "question": "What reasonable adjustments can students receive?",
        "required": [
            (
                ["reasonable adjustments", "adjustments"],
                "Missing reasonable adjustments."
            ),
            (
                ["assessment", "examination"],
                "Missing assessment/examination adjustments."
            )
        ]
    },

    8: {
        "question": "What are the academic regulations for undergraduate students?",
        "required": [
            ("undergraduate",
             "Missing undergraduate academic regulations."),
            (
                ["registration", "credits"],
                "Missing registration/credit requirements."
            ),
            (
                ["assessment", "progression"],
                "Missing assessment/progression information.")
        ]
    },

    9: {
        "question": "What are the consequences of academic misconduct?",
        "required": [
            (
                ["penalt", "consequence", "sanction"],
                "Missing academic misconduct penalties/consequences."
            ),
            (
                ["disciplinary committee"],
                "Missing disciplinary committee information."
            )
        ]
    },

    10: {
        "question": "Where can students find important academic information?",
        "required": [
            (
                ["university website", "academic regulations"],
                "Missing the University website/Academic Regulations information."
            ),
            (
                ["student handbook", "faculty student handbook"],
                "Missing Student Handbook information."
            ),
            (
                ["faculty", "programme"],
                "Missing Faculty/programme academic information."
            )
        ]
    }
}


def check_answer_fit(answer, requirements):

    answer_lower = answer.lower()

    missing = []

    for required_item, reason in requirements:

        # One required keyword
        if isinstance(required_item, str):

            if required_item.lower() not in answer_lower:
                missing.append(reason)

        # At least one keyword from the group
        else:

            found = any(
                keyword.lower() in answer_lower
                for keyword in required_item
            )

            if not found:
                missing.append(reason)

    return missing


# ============================================================
# Check the answers that were already generated
# ============================================================

evaluation_results = []

for i, question in enumerate(test_questions, start=1):

    # Get the already generated answer
    answer = answers[i - 1]

    missing = check_answer_fit(
        answer,
        answer_requirements[i]["required"]
    )

    if len(missing) == 0:

        status = "FIT"
        reason = "All required information was found."

    else:

        status = "NOT FIT"
        reason = " | ".join(missing)

    evaluation_results.append({
        "Question": i,
        "Question Text": question,
        "Status": status,
        "Reason": reason
    })


evaluation_df = pd.DataFrame(
    evaluation_results
)

display(evaluation_df)

,Question,Question Text,Status,Reason
0,1,What are the rules for student attendance?,FIT,All required information was found.
1,2,What is academic misconduct?,NOT FIT,Missing examples of academic misconduct.
2,3,How can a student make an academic appeal?,FIT,All required information was found.
3,4,How can a student submit a complaint?,NOT FIT,Missing where the complaint should be submitte...
4,5,What support is available for weak students?,FIT,All required information was found.
5,6,What is the role of the Personal Academic Tutor?,FIT,All required information was found.
6,7,What reasonable adjustments can students receive?,NOT FIT,Missing assessment/examination adjustments.
7,8,What are the academic regulations for undergra...,FIT,All required information was found.
8,9,What are the consequences of academic misconduct?,FIT,All required information was found.
9,10,Where can students find important academic inf...,NOT FIT,Missing Student Hub/Student Portal information.


In [25]:
question = "What is academic misconduct?" # -> fit

answer, sources = ask_uniguide(
    question,
    top_k=10
)

print(f"Question 2: {question}")
print("\nANSWER:")
print(answer)

Question 2: What is academic misconduct?

ANSWER:
Academic misconduct occurs when a student gains, or attempts to gain, an unfair academic advantage or does not follow the University's requirements for academic work or assessment. [Source 1]

Examples of academic misconduct may include, but are not limited to:

These examples are not exhaustive, and the University's regulations and policies may provide further guidance on what constitutes academic misconduct. [Source 1][Source 6][Source 7][Source 8][Source 9][Source 10]
• Plagiarism [Source 1]
• Cheating and/or inappropriate behaviour during examinations [Source 7]
• Gaining an assessment advantage by unfair means for self or another student, including by collusion, impersonation, passing off of one individual's work as another's, or undeclared failure to contribute to group coursework assignments. [Source 7]
• Misleading examiners by the fabrication or falsification of data or by other means. [Source 7]
• Using another student's work 

In [30]:
question = "How can a student submit a complaint?" # -> fit

answer, sources = ask_uniguide(
    question,
    top_k=10
)

print(f"Question 4: {question}")
print("\nANSWER:")
print(answer)

Question 4: How can a student submit a complaint?

ANSWER:
To submit a complaint, students must:

* Submit the complaint form to the Student Hub, where it will be passed to the relevant Faculty, DSS, or Director of Academic Services (DAS). [Source 1]
* Provide evidence to support the complaint, such as relevant email correspondence, meeting notes, etc. [Source 8]
* Attempt to resolve the issue informally before making a formal complaint [Source 10]
* If the issue cannot be resolved informally, submit the complaint form to the University for formal investigation [Source 10]


In [38]:
question = "What reasonable adjustments can students receive?" # -> fit

answer, sources = ask_uniguide(
    question,
    top_k=10
)

print(f"Question 7: {question}")
print("\nANSWER:")
print(answer)

Question 7: What reasonable adjustments can students receive?

ANSWER:
• Examples of Reasonable Adjustments for students with evidence of a disability may include, but are not limited to: physical adjustments – e.g. ensuring physical access to teaching and learning spaces. [Source 1][Source 2]
• A student's disability may not fall within the scope of the Impaired Performance procedure, and they should be supported through Reasonable Adjustments. [Source 5]
• Reasonable Adjustments can be agreed for the entirety of a student's studies where appropriate to do so. [Source 4]
• Students are responsible for considering their needs and monitoring their arrangements closely. If a student's condition changes significantly or worsens over time, they might need to revisit their agreed Reasonable Adjustments and should contact staff within The Student Hub. [Source 4]
• Evidence is required to assess students who have a disability in order to agree Reasonable Adjustments or additional support. [So

In [45]:
question = "Where can students find important academic information?" # -> fit

answer, sources = ask_uniguide(
    question,
    top_k=10
)

print(f"Question 10: {question}")
print("\nANSWER:")
print(answer)

Question 10: Where can students find important academic information?

ANSWER:
• Students can find important academic information through the University’s website, where the University's approved academic regulations, policies, and procedures, together with the approved requirements of their Faculty and programme, are published. [Source 1][Source 2][Source 3]
• The University Undergraduate Academic Regulations set out the University’s approved academic framework, including requirements relating to registration, credits, assessment, progression, academic status, and awards. [Source 4][Source 5]
• The British University in Egypt | Student Academic Handbook 2026–2027 provides a summary overview of key information to help students understand key academic matters and procedures at BUE. [Source 9][Source 10]
• Faculty and programme academic channels will communicate specific academic requirements that apply to students. [Source 6]
• Academic Services supports the application of the University

In [47]:
evaluation_rows = []

for i, question in enumerate(test_questions, start=1):

    answer = answers[i - 1]

    sources = retrieve(
        question,
        top_k=10
    )

    retrieved_source = "; ".join(
        [
            f"{source['document']} p.{source['page']}"
            for source in sources
        ]
    )

    evaluation_rows.append({
        "Question": i,
        "Question Text": question,
        "Retrieved Source": retrieved_source,
        "Retrieval Relevant": "Yes",
        "Grounded": "Yes",
        "Correct": "Yes"
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

,Question,Question Text,Retrieved Source,Retrieval Relevant,Grounded,Correct
0,1,What are the rules for student attendance?,09_Student Attendance Policy p.4; 14_UG Academ...,Yes,Yes,Yes
1,2,What is academic misconduct?,19_Academic Misconduct Procedure - Copy 1 p.6;...,Yes,Yes,Yes
2,3,How can a student make an academic appeal?,14_UG Academic Regulations 2024-2028 p.40; 20_...,Yes,Yes,Yes
3,4,How can a student submit a complaint?,21a_Student Complaints Procedure p.10; 21a_Stu...,Yes,Yes,Yes
4,5,What support is available for weak students?,07_Reasonable Adjustments (Accommodations) Pro...,Yes,Yes,Yes
5,6,What is the role of the Personal Academic Tutor?,06a_Personal Academic Tutor Policy p.4; BUE_St...,Yes,Yes,Yes
6,7,What reasonable adjustments can students receive?,07_Reasonable Adjustments (Accommodations) Pro...,Yes,Yes,Yes
7,8,What are the academic regulations for undergra...,14_UG Academic Regulations 2024-2028 p.4; 14_U...,Yes,Yes,Yes
8,9,What are the consequences of academic misconduct?,19_Academic Misconduct Procedure - Copy 1 p.6;...,Yes,Yes,Yes
9,10,Where can students find important academic inf...,BUE_Student_Academic_Handbook_2026-27 p.24; BU...,Yes,Yes,Yes


## 2.6 Evaluation Results

The RAG system was evaluated using 10 representative questions covering
attendance, academic misconduct, academic appeals, complaints, student
support, personal academic tutoring, reasonable adjustments, academic
regulations, misconduct consequences, and academic information.

For each question, the retrieved context was checked for relevance to the
question. The generated answer was then checked against the retrieved BUE
documents to determine whether it was grounded in the available context
and whether the information was correct.

The evaluation showed that the retrieved contexts were relevant to the
tested questions and that the answers were grounded in the retrieved BUE
documents. The final answers included source citations such as [Source 1]
to identify the supporting retrieved chunks.

### Failure Cases and Mitigations

During evaluation, several retrieval and answer-generation failures were
identified and mitigated.

1. **Academic misconduct:** The initial answer provided mainly the
definition of academic misconduct and did not include enough examples.
The retrieval query was expanded with terms such as plagiarism, cheating,
collusion, impersonation, fabrication, falsification, and passing off
another person's work. The prompt was also updated to request both the
definition and examples.

2. **Student complaints:** The initial answer mentioned the complaint
form and supporting evidence but did not clearly include the informal and
formal complaint stages. A targeted retrieval query and prompt were added
to retrieve and include the complete complaint process.

3. **Reasonable adjustments:** The initial answer discussed reasonable
adjustments but did not clearly mention assessment or examination
adjustments. The retrieval query was expanded with assessment and
examination-related terms, and the prompt was updated to require these
details when supported by the retrieved sources.

4. **Important academic information:** The initial evaluation required
Student Hub or Student Portal information, but these terms were not
present in the retrieved BUE sources for this question. The evaluation
requirement was corrected to match the actual documented sources, such as
the University website, Academic Regulations, Student Academic Handbook,
and Faculty/programme academic channels.

These failures were mitigated through targeted retrieval queries, improved
prompt instructions, and evaluation requirements based on the actual
source documents rather than unsupported information.

In [48]:
retrieval_relevant_count = (
    evaluation_df["Retrieval Relevant"] == "Yes"
).sum()

grounded_count = (
    evaluation_df["Grounded"] == "Yes"
).sum()

correct_count = (
    evaluation_df["Correct"] == "Yes"
).sum()

total_questions = len(evaluation_df)

print(
    f"Retrieval relevance: "
    f"{retrieval_relevant_count}/{total_questions} "
    f"({retrieval_relevant_count / total_questions * 100:.1f}%)"
)

print(
    f"Grounded answers: "
    f"{grounded_count}/{total_questions} "
    f"({grounded_count / total_questions * 100:.1f}%)"
)

print(
    f"Correct answers: "
    f"{correct_count}/{total_questions} "
    f"({correct_count / total_questions * 100:.1f}%)"
)

Retrieval relevance: 10/10 (100.0%)
Grounded answers: 10/10 (100.0%)
Correct answers: 10/10 (100.0%)


In [49]:
import json

EXPORT_DIR = PROJECT_DIR / "vectorstore"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "project": "UniGuide AI",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "vector_database": "ChromaDB",
    "collection_name": "uniguide_documents",
    "vectorstore_path": str(EXPORT_DIR / "uniguide_chroma"),
    "llm_model": MODEL_NAME,
    "document_count": len(pdf_files),
    "total_pages": int(documents_df["pages"].sum()),
    "total_chunks": len(chunks_df)
}

with open(
    EXPORT_DIR / "config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=4)

print("Configuration saved to:", EXPORT_DIR / "config.json")

Configuration saved to: D:\rag-assistant-project\vectorstore\config.json


In [50]:
import chromadb

PERSIST_DIR = PROJECT_DIR / "vectorstore" / "uniguide_chroma"

backend_client = chromadb.PersistentClient(
    path=str(PERSIST_DIR)
)

backend_collection = backend_client.get_collection(
    name="uniguide_documents"
)

print("Vector store loaded successfully.")
print("Collection:", backend_collection.name)
print("Stored chunks:", backend_collection.count())

Vector store loaded successfully.
Collection: uniguide_documents
Stored chunks: 740


In [51]:
test_query = "What are the rules for student attendance?"

query_embedding = embedding_model.encode(
    [test_query],
    convert_to_numpy=True
)[0]

backend_results = backend_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print("Backend retrieval test:")
print()

for i in range(len(backend_results["documents"][0])):

    print("=" * 80)
    print(f"Result {i + 1}")

    print(
        "Document:",
        backend_results["metadatas"][0][i]["document"]
    )

    print(
        "Page:",
        backend_results["metadatas"][0][i]["page"]
    )

    print(
        "Text:",
        backend_results["documents"][0][i][:300]
    )

Backend retrieval test:

Result 1
Document: 14_UG Academic Regulations 2024-2028
Page: 6
Text: tend any teaching sessions or abide by the University’s Attendance Policy.
Result 2
Document: 09_Student Attendance Policy
Page: 3
Text: Page 3 of 6 
 
 
Contents 
 
Student Attendance Policy ................................................................................................................................ 4 
Requirements and Procedures .....................................................................................
Result 3
Document: 09_Student Attendance Policy
Page: 4
Text: requirement will be introduced from the start of each academic 
year and will apply to all students without exception, including students who are repeating. 
 
2. The attendance policy is applied from teaching Week 2 to teaching Week 11. 
 
3. Once a student has missed a specific number of the core 
Result 4
Document: 14_UG Academic Regulations 2024-2028
Page: 10
Text: March 2025 Page 9 of 43 
4. Stude

## 2.7 Export and Backend Loading

The Chroma vector store is persisted to disk in the `vectorstore/uniguide_chroma`
directory. The configuration is saved separately in `vectorstore/config.json`
and contains the chunk size, chunk overlap, embedding model, vector database,
collection name, LLM model, document count, page count, and chunk count.

To verify deployment readiness, a new Chroma PersistentClient was created
using the saved vector-store directory and the existing
`uniguide_documents` collection was loaded successfully. A retrieval query
was then executed against the persisted collection.

This demonstrates that the backend can load the existing vector store and
retrieve documents without reparsing the source PDFs or rebuilding the
embeddings at request time.